# VietTDR — vòng 2: đánh giá lại + huấn luyện ngân sách lớn (MỘT phiên)

## Vòng 1 đã cho gì

| Cấu hình | word acc | 1−NED | Δ word |
|---|---|---|---|
| full | 0.7267 | 0.8571 | — |
| flat (bỏ phân rã) | 0.7134 | 0.8502 | −1.33 |
| no_toneprior | 0.6693 | 0.8314 | −5.74 |
| no_comploss | 0.6886 | 0.8372 | −3.81 |

Cả ba điểm mới đều dương. Vòng 2 sửa hai vấn đề còn lại:

1. Chỉ số `diacritic-heavy` cũ chỉ có n=62 nên vô dụng → thay bằng bốn tập
   con đúng cỡ: `any_diac` 61.5%, `multi_diac` 7.9%, `confusable` 9.2%,
   `stacked` 23.7%.
2. Pretrain 6 epoch là quá ngắn — word acc nhảy từ 0.008 ở cuối epoch 1 của
   pretrain lên 0.57 ngay epoch 2 của fine-tune, tức là nó vẫn đang lên dốc
   khi bị cắt → vòng 2 chạy 15 epoch trên 500k ảnh.

## Ngân sách — chạy trọn MỘT phiên

| Việc | Thời gian |
|---|---|
| Cell 3: đánh giá lại 4 checkpoint vòng 1 | ~0.3 h |
| Cell 4: sinh 500k ảnh tổng hợp | ~0.25 h |
| Cell 5: 4 cấu hình × (pretrain 15 ep + fine-tune 30 ep) | ~6.3 h |
| **Tổng** | **~6.9 h** |

Ước tính dựa trên tốc độ **đo được** ở vòng 1 (508 ảnh/s với augmentation
đầy đủ), nhân hệ số 5.2× của chế độ `--synth-aug light`.

> **Chốt chặn an toàn.** Cell 1 đặt đồng hồ `BUDGET_H = 10.0`. Trước mỗi cấu
> hình, cell 5 kiểm tra thời gian đã chạy; vượt mốc thì **bỏ qua phần còn
> lại** và đi thẳng xuống đóng gói. Lý do: giới hạn cứng của Kaggle là 12h,
> phiên quá giờ bị đánh dấu thất bại và kết quả trong `/kaggle/working` có
> thể **mất trắng**. Bốn cấu hình xếp theo độ quan trọng giảm dần nên nếu
> thiếu giờ thì mất cái ít quan trọng nhất; ngoài ra mỗi cấu hình được đánh
> giá ngay sau khi train xong chứ không dồn đến cuối.

## Chuẩn bị: Add Input bốn nguồn

| Nguồn | Dùng cho |
|---|---|
| `viettdr-data` | ảnh từ đã cắt |
| dataset code mới (tên có chữ `code`) | mã nguồn — phải là bản mới nhất |
| `vintext-train-images` | nền thật cho ảnh tổng hợp (~35% số mẫu) |
| Notebook Output của vòng 1 | checkpoint cho cell 3 |

Settings: GPU **T4 x2**. Internet không cần (font đã nằm trong zip).

## Khác vòng 1 ở đâu

1. Pretrain 15 epoch × 500k ảnh (vòng 1: 6 epoch × 200k).
2. Ảnh tổng hợp khớp phân bố độ phân giải thật nhờ `--height-ref`:
   p10/p50/p90 = 12/25/88 px so với ảnh thật 13/29/110. Vòng 1 là 25/37/50,
   tức là thiếu hẳn nhóm ảnh nhỏ mờ — đúng nhóm khó nhất.
3. Nền cắt từ ảnh cảnh gốc VinText (~35% số mẫu).
4. Fine-tune trộn thêm 50k ảnh tổng hợp để chống overfit. Vòng 1 loss huấn
   luyện đã chạm sàn (0.838 so với sàn lý thuyết 0.776) nhưng val khựng ở
   71.8% — dấu hiệu overfit trên 25k ảnh thật.
5. `--synth-aug light`: bỏ augmentation trùng lặp trên ảnh tổng hợp (chúng
   đã có sẵn mờ/nhiễu/xoay/JPEG từ lúc sinh) → phần nạp dữ liệu nhanh 5.2×,
   đo được 4.50 ms/ảnh xuống 0.87 ms/ảnh.


In [ ]:
# Cell 1 — moi truong
import torch, os, multiprocessing
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '*** KHONG CO GPU - bat Accelerator ***')
print('CPU   :', multiprocessing.cpu_count(), 'cores')
IS_KAGGLE = os.path.exists('/kaggle')
NPROC = max(2, multiprocessing.cpu_count())

import time
T0 = time.time()
BUDGET_H = 10.0   # dung khoi dong cau hinh moi neu da qua moc nay
                  # (gioi han cung cua Kaggle la 12h; qua gio = MAT TRANG ket qua)
def elapsed_h(): return (time.time() - T0) / 3600
print(f'moc bat dau phien da dat, ngan sach {BUDGET_H}h')

In [ ]:
# Cell 2 — nap code + du lieu (giong vong 1)
import os, glob, shutil, time

# WORK phai nam NGOAI /kaggle/working: Kaggle gioi han so file cua output,
# ma synth 500k + crops 41k lam no bo LUON ca output (ca hai lan chay truoc
# deu mat sach checkpoint vi ly do nay, du log ghi da tao xong file zip).
# Chi ket qua cuoi moi duoc chep sang /kaggle/working o cell 6.
SRC, WORK = ('/kaggle/input', '/tmp/vt') if IS_KAGGLE else \
            ('/content/drive/MyDrive', '/content/run')
OUTDIR = '/kaggle/working' if IS_KAGGLE else '.'
os.makedirs(WORK, exist_ok=True)

def find_all(root, name):
    return sorted(glob.glob(f'{root}/**/{name}', recursive=True))

def pick(hits, prefer='code'):
    if not hits:
        return None
    for h in hits:
        if prefer in h.lower():
            return h
    return hits[0]

# Chon ban code theo NANG LUC, khong theo ten/thu tu chu cai: tai khoan co
# nhieu dataset chua chu 'code' va sorted() se lay ban CU truoc.
NEEDED = ['--synth-aug', '--init-from', '--no-decompose']
cands = find_all(SRC, 'train.py')
good = []
for c in cands:
    try:
        t = open(c, encoding='utf-8').read()
        if all(f in t for f in NEEDED):
            good.append(c)
    except Exception:
        pass
print('ban code tim thay:')
for c in cands:
    print(('   [DU CO] ' if c in good else '   [CU]    ') + os.path.dirname(c))
assert good, ('Khong ban code nao co du ' + str(NEEDED) +
              ' -> upload lai VietTDR.zip moi')
train_py = good[-1]
code_root = os.path.dirname(train_py)
for item in ['viettdr', 'tools', 'tests', 'fonts', 'assets', 'train.py',
             'eval.py', 'charset_vintext.txt']:
    s, d = os.path.join(code_root, item), os.path.join(WORK, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.isfile(s):
        shutil.copy(s, d)

gt = pick(find_all(SRC, 'train_gt.jsonl'), 'vintext')
assert gt, 'Khong tim thay train_gt.jsonl'
data_dst = os.path.join(WORK, 'data', 'vintext_words')
if not os.path.exists(os.path.join(data_dst, 'train_gt.jsonl')):
    t0 = time.time()
    os.makedirs(os.path.dirname(data_dst), exist_ok=True)
    shutil.copytree(os.path.dirname(gt), data_dst)
    print(f'copy du lieu -> o local: {time.time() - t0:.0f}s')

os.chdir(WORK)
src = open('train.py', encoding='utf-8').read()
met = open('viettdr/metrics.py', encoding='utf-8').read()
for flag, hay in [('--synth-aug', src), ('--init-from', src),
                  ('--no-decompose', src), ('confusable', met),
                  ('stacked', met)]:
    print(('  [OK]    ' if flag in hay else '  [THIEU] ') + flag)
print('\ncode:', code_root)

In [ ]:
# Cell 3 — DANH GIA LAI 4 checkpoint cua vong 1 voi 4 tap con moi (~20 phut)
# Can Add Input = Notebook Output cua lan chay vong 1.
import glob, os, shutil

os.makedirs('runs_v1', exist_ok=True)
# neu output vong 1 chi co file zip (chua duoc giai nen) thi bung ra truoc
for z in glob.glob('/kaggle/input/**/results_viettdr*.zip', recursive=True):
    get_ipython().system(f'unzip -qo "{z}" -d runs_v1_src')
found = {}
for name in ['full', 'flat', 'no_toneprior', 'no_comploss']:
    pool = (glob.glob('/kaggle/input/**/*.pth', recursive=True)
            + glob.glob('runs_v1_src/**/*.pth', recursive=True))
    hits = [p for p in pool
            if f'/{name}/' in p.replace(os.sep, '/')
            and 'pre_' not in p.replace(os.sep, '/')]
    # uu tien best_slim.pth roi den best.pth
    hits.sort(key=lambda p: ('best_slim' not in p, 'best' not in p))
    if hits:
        found[name] = hits[0]
print('checkpoint tim thay:')
for k, v in found.items():
    print(f'  {k:14} {v}')
if not found:
    print('  (chua Add Input Notebook Output cua vong 1 -> bo qua cell nay)')

for name, ck in found.items():
    print('=' * 22, name, '=' * 22, flush=True)
    !python eval.py --ckpt {ck} --data data/vintext_words --split test \
        --charset charset_vintext.txt --out runs_v1/{name}_eval.json

In [ ]:
# Cell 4 — sinh 500k anh tong hop (~15 phut)
#
# --height-ref: lay mau chieu cao tu chinh anh that cua VinText.
# Do duoc: anh that cao p10=13 p50=29 p90=110 px, con bo sinh voi co chu
# co dinh chi cho 25-50 px -> model khong bao gio thay anh nho mo, dung
# cai kho nhat. Sau khi lay mau: p10=12 p50=25 p90=88, khop han.
N_SYNTH = 500000
N_SYNTH_FT = 50000        # tap con nho tron vao giai doan 2 de chong overfit

import os, json, random, glob
corpus = ['data/vintext_words/train_gt.jsonl']
if os.path.exists('assets/general_dict.txt'):
    corpus.append('assets/general_dict.txt')
corpus = ' '.join(corpus)

# nen that: dataset vintext-train-images (anh canh goc VinText)
bg_hits = glob.glob('/kaggle/input/**/im0001.jpg', recursive=True)
BG = ('--bg-dir ' + os.path.dirname(bg_hits[0])) if bg_hits else ''
print('nen that:', os.path.dirname(bg_hits[0]) if bg_hits else
      'KHONG TIM THAY -> dung 100% nen tu tao (van chay duoc, nen Add Input vintext-train-images)')

if not os.path.exists('data/synth/synth_gt.jsonl'):
    !python tools/gen_synth.py --out data/synth --count {N_SYNTH} \
        --fonts fonts --corpus {corpus} --workers {NPROC} \
        --height-ref data/vintext_words/train_gt.jsonl \
        --height-ref-dir data/vintext_words/train \
        --bg-cache 40 --tone-balance {BG}

# tap con cho fine-tune
lines = open('data/synth/synth_gt.jsonl', encoding='utf-8').read().splitlines()
random.seed(0)
sub = random.sample(lines, min(N_SYNTH_FT, len(lines)))
open('data/synth/synth_ft.jsonl', 'w', encoding='utf-8').write('\n'.join(sub) + '\n')
print(f'tong hop: {len(lines)} anh | tap con cho FT: {len(sub)}')

# kiem tra du 6 thanh dieu
import sys, collections
sys.path.insert(0, '.')
from viettdr.vietchar import decompose_char
tones = collections.Counter()
for line in lines[:20000]:
    for ch in json.loads(line)['text']:
        tones[decompose_char(ch)[2]] += 1
n = sum(tones.values())
print({t: f'{100*v/n:.1f}%' for t, v in sorted(tones.items())})
assert len(tones) == 6, 'THIEU THANH DIEU!'

In [ ]:
# Cell 5 — HUAN LUYEN CA 4 CAU HINH TRONG MOT PHIEN (~6.5h)
#
# Xep theo do quan trong giam dan: neu het gio thi mat cau hinh it quan
# trong nhat. Truoc moi cau hinh co kiem tra dong ho: da qua BUDGET_H thi
# BO QUA phan con lai va di thang xuong cell 6 de con kip dong goi ket qua.
# Ly do: Kaggle cat cung o 12h va phien qua gio bi danh dau that bai,
# ket qua trong /kaggle/working co the KHONG duoc luu.

CONFIGS = [('no_toneprior', '--no-tone-prior')]   # (2) tone prior
# 'full' va 'flat' da xong o version 2 (word acc 0.8196 / 0.8194).
# 'no_comploss' de tuan sau khi quota GPU reset (08-08).

import os
PRE = (f'--data data/vintext_words --charset charset_vintext.txt '
       f'--synth-jsonl data/synth/synth_gt.jsonl --synth-dir data/synth '
       f'--synth-aug light --epochs 15 --bs 192 --workers {NPROC} '
       f'--eval-every 5 --val-subset 1500')
FT = (f'--data data/vintext_words --charset charset_vintext.txt '
      f'--synth-jsonl data/synth/synth_ft.jsonl --synth-dir data/synth '
      f'--synth-aug light --epochs 30 --bs 192 --workers {NPROC} '
      f'--eval-every 3 --val-subset 1500 --lr 2e-4')

def ckpt(run):
    for fn in ['best.pth', 'last.pth']:
        p = f'runs/{run}/{fn}'
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'runs/{run}: chua co checkpoint')

DONE, SKIPPED = [], []
for name, flag in CONFIGS:
    if elapsed_h() > BUDGET_H:
        SKIPPED.append(name); print(f'### BO QUA {name}: da chay {elapsed_h():.1f}h > {BUDGET_H}h'); continue
    print('#'*22, f'{name}  (da chay {elapsed_h():.2f}h)', '#'*22, flush=True)
    if not os.path.exists(f'runs/pre_{name}/last.pth'):
        !python train.py {PRE} --out runs/pre_{name} {flag}
    ck = ckpt('pre_' + name)
    !python train.py {FT} --out runs/{name} {flag} --init-from {ck}
    if os.path.exists(f'runs/{name}/best.pth'):
        DONE.append(name)
        # danh gia + dong goi NGAY sau moi cau hinh, khong doi den cuoi
        !python eval.py --ckpt runs/{name}/best.pth --data data/vintext_words --split test --charset charset_vintext.txt

print(f'{chr(10)}xong {DONE} | bo qua {SKIPPED} | tong {elapsed_h():.2f}h')

In [ ]:
# Cell 6 — danh gia tren test + dong goi
import os, json, shutil, torch

# (eval da chay ngay sau moi cau hinh o cell 5)

os.makedirs('/tmp/res', exist_ok=True)
summary = {}
for name in DONE:
    d = f'/tmp/res/{name}'
    os.makedirs(d, exist_ok=True)
    for f in ['log.csv', 'eval_test.json']:
        if os.path.exists(f'runs/{name}/{f}'):
            shutil.copy(f'runs/{name}/{f}', d)
    if os.path.exists(f'runs/pre_{name}/log.csv'):
        shutil.copy(f'runs/pre_{name}/log.csv', f'{d}/pretrain_log.csv')
    if os.path.exists(f'runs/{name}/eval_test.json'):
        s = json.load(open(f'runs/{name}/eval_test.json', encoding='utf-8'))['summary']
        summary[name] = {k: round(s[k], 4) for k in
                         ['word_acc', 'one_minus_ned', 'any_diac_acc',
                          'multi_diac_acc', 'confusable_acc', 'stacked_acc']}
    if os.path.exists(f'runs/{name}/best.pth'):
        c = torch.load(f'runs/{name}/best.pth', map_location='cpu',
                       weights_only=False)
        torch.save({'model': c['model'], 'args': c['args'],
                    'epoch': c['epoch'], 'best': c['best']}, f'{d}/best_slim.pth')

# kem ket qua danh gia lai cua vong 1 neu co
if os.path.isdir('runs_v1'):
    shutil.copytree('runs_v1', '/tmp/res/v1_reeval', dirs_exist_ok=True)

print(json.dumps(summary, indent=1, ensure_ascii=False))
json.dump(summary, open('/tmp/res/summary.json', 'w'), indent=1)
dest = OUTDIR
shutil.make_archive(f'{dest}/results_v2', 'zip', '/tmp/res')
print(f'\nXONG -> {dest}/results_v2.zip '
      f'({os.path.getsize(dest + "/results_v2.zip")/1e6:.0f} MB)')
# doi chieu: output chi duoc phep co vai file, nhieu la Kaggle bo het
n_out = sum(len(f) for _, _, f in os.walk(dest))
print(f'so file trong {dest}: {n_out}' + ('  [OK]' if n_out < 500 else '  [NGUY HIEM - Kaggle se bo output]'))
